In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils import shuffle
from sklearn import svm
import pyswarms as ps
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from deap import algorithms, base, creator, tools

# Read data
data = pd.read_excel('聚类分析2.xlsx')  # Please replace with your data file name
data = data.drop('合金的牌号', axis=1)
data = data.round(2)
elements = ['Ni', 'Cr', 'Co', 'Fe', 'Al', 'Ti', 'Nb', 'Mo', 'W', 'C', 'B', 'Zr', '温度', '应力', '固溶处理温度',
            '固溶处理时间', '强化相溶解温度', '稳定时效温度', '稳定时效时间', '时效温度', '时效时间']

X = np.zeros((data.shape[0], len(elements)))
for i, row in data.iterrows():
    for j, el in enumerate(elements):
        X[i, j] = row[el]
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
Y = data[['蠕变时间']].values
Y = np.log(Y)


def evaluate(true_labels, pred_labels):
    errors = abs(pred_labels - true_labels)
    MAE = mean_absolute_error(true_labels, pred_labels)
    mape = 100 * np.mean(errors / true_labels)
    r2 = r2_score(true_labels, pred_labels)
    RMSE = mean_squared_error(true_labels, pred_labels, squared=False)
    return mape, MAE, RMSE, r2


model_0 = SVR(C=37.722551111111111111, kernel='rbf', gamma=0.111111111111111111)
cv = LeaveOneOut()
Model = model_0
X = X.reshape(-1, len(elements))
Model.fit(X, Y)
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(Model, X, Y, cv=cv)
model_accuracy1, model_MAE1, model_RMSE1, model_r21 = evaluate(Y, y_pred)
print("R^2 score:", model_r21)


def fitness_func(individual, X, Y, model):
    X_individual = np.tile(individual, (X.shape[0], 1))
    Y_pred = model.predict(X_individual)
    mse = mean_squared_error(Y, Y_pred)
    return (1 / (1 + mse),)


# Define GA parameters
POP_SIZE = 400
NUM_GEN = 1
CXPB = 0.9
MUTPB = 0.01

# Create a fitness function for maximizing fitness values
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Initialize toolbox
toolbox = base.Toolbox()

# Define attributes
toolbox.register("attr_float", np.random.uniform, low=0, high=1)

# Define individual and population
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=len(elements))
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Define evaluation function
toolbox.register("evaluate", fitness_func, X=X, Y=Y, model=Model)

# Define genetic operators
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)


# Perform genetic algorithm
population = toolbox.population(n=POP_SIZE)
hall_of_fame = tools.HallOfFame(maxsize=1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)

results = []
for _ in range(40):
    population, logbook = algorithms.eaSimple(population, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=NUM_GEN,
                                              stats=stats, halloffame=hall_of_fame, verbose=True)

    # Extract best individual and its fitness
    best_individual = hall_of_fame[0]
    best_fitness = best_individual.fitness.values[0]

    # Reshape best_individual
    best_particles = np.asarray(best_individual)

    results.append((best_particles, elements))

# Create DataFrame for storing results
df = pd.DataFrame(results, columns=['Best Particles', 'Elements'])
df.to_excel('results.xlsx', index=False)

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

R^2 score: 0.8804390625989749
gen	nevals	avg     	min      
0  	400   	0.331171	0.0824012
1  	376   	0.374914	0.174437 
2  	371   	0.383292	0.162068 
3  	362   	0.383359	0.189632 
4  	368   	0.387544	0.192026 
5  	356   	0.384116	0.151593 
6  	374   	0.38961 	0.21903  
7  	358   	0.394675	0.198717 
8  	364   	0.393605	0.165955 
9  	366   	0.395195	0.190385 
10 	348   	0.394958	0.18023  
11 	366   	0.392584	0.202961 
12 	362   	0.40061 	0.209199 
13 	360   	0.403593	0.296796 
14 	370   	0.40375 	0.24022  
15 	356   	0.404286	0.216048 
16 	372   	0.405001	0.223385 
17 	355   	0.409592	0.248229 
18 	354   	0.408789	0.263982 
19 	350   	0.41021 	0.240794 
20 	354   	0.408553	0.151297 
21 	367   	0.40905 	0.314072 
22 	368   	0.409088	0.355709 
23 	364   	0.408297	0.322934 
24 	357   	0.408741	0.265829 
25 	346   	0.410179	0.345641 
26 	350   	0.409782	0.32503  
27 	370   	0.410191	0.352558 
28 	363   	0.411962	0.366772 
29 	368   	0.412333	0.268131 
30 	348   	0.413408	0.281006 
31 	356   

74 	343   	0.416256	0.397202
75 	374   	0.416342	0.416342
76 	374   	0.415616	0.310549
77 	360   	0.415708	0.175657
78 	376   	0.41602 	0.288931
79 	357   	0.416341	0.415861
80 	352   	0.416304	0.401142
81 	356   	0.416323	0.41056 
82 	349   	0.416342	0.416342
83 	361   	0.416147	0.357705
84 	368   	0.416067	0.306283
85 	357   	0.41542 	0.236979
86 	356   	0.415404	0.222429
87 	376   	0.416246	0.379358
88 	348   	0.415957	0.262665
89 	354   	0.416076	0.309938
90 	370   	0.41617 	0.372563
91 	366   	0.416342	0.416342
92 	366   	0.415811	0.204029
93 	366   	0.415907	0.264366
94 	365   	0.41634 	0.415584
95 	360   	0.416178	0.357705
96 	368   	0.416276	0.397471
97 	358   	0.415621	0.234164
98 	341   	0.415973	0.275287
99 	354   	0.416186	0.374601
100	371   	0.416064	0.312015
gen	nevals	avg     	min     
0  	0     	0.416064	0.312015
1  	352   	0.416327	0.41056 
2  	354   	0.416337	0.414818
3  	344   	0.416075	0.309938
4  	369   	0.41528 	0.226236
5  	358   	0.416131	0.357705
6  	364   	0.4

51 	363   	0.416275	0.389539
52 	360   	0.41634 	0.415429
53 	350   	0.416093	0.322539
54 	372   	0.416189	0.379358
55 	351   	0.416341	0.415861
56 	364   	0.416342	0.416342
57 	360   	0.416342	0.416342
58 	352   	0.415893	0.23657 
59 	358   	0.416249	0.379031
60 	372   	0.416182	0.382318
61 	374   	0.41611 	0.357705
62 	367   	0.41611 	0.324986
63 	373   	0.415948	0.264366
64 	364   	0.415686	0.289581
65 	343   	0.415932	0.309938
66 	360   	0.416295	0.397471
67 	362   	0.416244	0.385327
68 	348   	0.414878	0.125819
69 	363   	0.416157	0.379358
70 	346   	0.416342	0.416342
71 	343   	0.416025	0.309938
72 	370   	0.416342	0.416208
73 	350   	0.416342	0.416342
74 	359   	0.41589 	0.235861
75 	354   	0.416321	0.40818 
76 	360   	0.415819	0.265986
77 	364   	0.415513	0.23657 
78 	339   	0.416332	0.413616
79 	368   	0.416275	0.397471
80 	358   	0.415915	0.264366
81 	342   	0.416113	0.324528
82 	368   	0.416328	0.41056 
83 	358   	0.416304	0.401588
84 	351   	0.415408	0.174586
85 	367   	0.4

28 	369   	0.416339	0.415095
29 	361   	0.416029	0.292508
30 	366   	0.416007	0.323279
31 	372   	0.416338	0.414818
32 	370   	0.416325	0.409497
33 	367   	0.415893	0.23657 
34 	350   	0.415547	0.251016
35 	352   	0.416342	0.416342
36 	364   	0.416326	0.41056 
37 	367   	0.415811	0.206862
38 	357   	0.415972	0.273911
39 	364   	0.416326	0.41056 
40 	372   	0.415946	0.263537
41 	367   	0.415893	0.23657 
42 	372   	0.415871	0.23657 
43 	348   	0.415958	0.263049
44 	361   	0.415888	0.23657 
45 	359   	0.415962	0.264366
46 	346   	0.416295	0.397471
47 	358   	0.416245	0.397202
48 	358   	0.416055	0.309938
49 	365   	0.415807	0.302338
50 	360   	0.416195	0.357705
51 	358   	0.416329	0.41218 
52 	365   	0.416188	0.361675
53 	362   	0.416342	0.416188
54 	346   	0.416342	0.416337
55 	359   	0.415469	0.314441
56 	361   	0.415545	0.250151
57 	353   	0.416339	0.415095
58 	368   	0.416258	0.382651
59 	348   	0.415504	0.23657 
60 	365   	0.415706	0.254091
61 	369   	0.416241	0.395524
62 	362   	0.4

...

71 	367   	0.416325	0.409497
72 	350   	0.415305	0.23657 
73 	360   	0.416341	0.415825
74 	362   	0.41632 	0.40818 
75 	348   	0.416342	0.416342
76 	370   	0.416319	0.40818 
77 	352   	0.416175	0.357705
78 	352   	0.416342	0.416276
79 	346   	0.415956	0.264366
80 	355   	0.41634 	0.415263
81 	364   	0.416288	0.394587
82 	356   	0.415928	0.257685
83 	364   	0.415841	0.23657 
84 	363   	0.415696	0.264366
85 	368   	0.41596 	0.264366
86 	360   	0.416146	0.357705
87 	356   	0.416335	0.41362 
88 	365   	0.415577	0.262041
89 	364   	0.416033	0.309938
90 	345   	0.41625 	0.397202
91 	336   	0.416148	0.33867 
92 	362   	0.416342	0.416307
93 	343   	0.415683	0.242994
94 	370   	0.415962	0.264366
95 	355   	0.416238	0.374568
96 	364   	0.416338	0.414818
97 	358   	0.416341	0.415861
98 	366   	0.416295	0.397471
99 	376   	0.415651	0.261886
100	372   	0.41563 	0.238851
gen	nevals	avg    	min     
0  	0     	0.41563	0.238851
1  	360   	0.416069	0.308219
2  	366   	0.416342	0.416337
3  	366   	0.416

48 	363   	0.416076	0.309938
49 	360   	0.416091	0.357705
50 	372   	0.416341	0.415861
51 	348   	0.416341	0.415831
52 	359   	0.416341	0.415825
53 	348   	0.415659	0.309938
54 	355   	0.415324	0.264366
55 	367   	0.416341	0.415861
56 	361   	0.416255	0.392904
57 	360   	0.416094	0.325619
58 	354   	0.415484	0.23657 
59 	365   	0.416002	0.280719
60 	340   	0.415962	0.264366
61 	366   	0.416297	0.398225
62 	364   	0.415958	0.264366
63 	358   	0.415137	0.163532
64 	358   	0.415986	0.273755
65 	344   	0.416307	0.404765
66 	372   	0.416148	0.357705
67 	368   	0.415891	0.23657 
68 	364   	0.41578 	0.23657 
69 	359   	0.416342	0.416328
70 	368   	0.416306	0.401841
71 	366   	0.415687	0.174586
72 	353   	0.415962	0.265437
73 	362   	0.416148	0.357705
74 	370   	0.415913	0.293841
75 	364   	0.415596	0.264366
76 	350   	0.416339	0.415095
77 	368   	0.416105	0.370987
78 	344   	0.416342	0.416188
79 	358   	0.416338	0.414818
80 	361   	0.416076	0.309938
81 	354   	0.415777	0.264366
82 	358   	0.4

25 	377   	0.416276	0.397202
26 	360   	0.415947	0.304411
27 	362   	0.416251	0.388527
28 	362   	0.416342	0.416342
29 	354   	0.416237	0.383079
30 	364   	0.416342	0.416342
31 	365   	0.416329	0.410978
32 	365   	0.416276	0.392808
33 	344   	0.415856	0.221856
34 	374   	0.416292	0.39647 
35 	359   	0.416336	0.414818
36 	372   	0.416322	0.40818 
37 	364   	0.416342	0.416337
38 	370   	0.416249	0.379358
39 	358   	0.415959	0.264366
40 	348   	0.416059	0.314719
41 	363   	0.416146	0.377694
42 	348   	0.416076	0.309938
43 	359   	0.416192	0.357705
44 	365   	0.416195	0.357705
45 	368   	0.416342	0.416342
46 	362   	0.41625 	0.397202
47 	371   	0.416295	0.397471
48 	372   	0.415071	0.174586
49 	373   	0.415892	0.23657 
50 	356   	0.415861	0.23657 
51 	360   	0.416075	0.309938
52 	362   	0.416209	0.379358
53 	370   	0.416208	0.379358
54 	361   	0.416341	0.415825
55 	365   	0.41609 	0.322699
56 	359   	0.416213	0.372999
57 	356   	0.415893	0.23657 
58 	362   	0.41634 	0.415263
59 	349   	0.4

2  	366   	0.41558 	0.218456
3  	362   	0.416342	0.416342
4  	360   	0.415974	0.268993
5  	354   	0.416321	0.40818 
6  	370   	0.415425	0.189604
7  	380   	0.416201	0.379358
8  	371   	0.415642	0.280719
9  	357   	0.415997	0.282051
10 	351   	0.41634 	0.415861
11 	363   	0.415612	0.263049
12 	360   	0.416342	0.416342
13 	371   	0.416257	0.390762
14 	348   	0.416016	0.312572
15 	354   	0.415796	0.23657 
16 	371   	0.416339	0.415095
17 	361   	0.416321	0.41056 
18 	360   	0.416339	0.415095
19 	356   	0.415842	0.309938
20 	378   	0.41625 	0.379358
21 	363   	0.416277	0.392257
22 	354   	0.415675	0.151982
23 	342   	0.416342	0.416158
24 	368   	0.416328	0.41056 
25 	344   	0.41634 	0.415272
26 	358   	0.416311	0.40818 
27 	365   	0.415758	0.201577
28 	356   	0.415795	0.354227
29 	354   	0.415359	0.221856
30 	354   	0.415893	0.23657 
31 	362   	0.416295	0.397471
32 	372   	0.416342	0.416307
33 	355   	0.416337	0.414818
34 	356   	0.416338	0.414818
35 	350   	0.416228	0.379358
36 	353   	0.4

81 	358   	0.416076	0.309938
82 	378   	0.416291	0.397202
83 	358   	0.416056	0.309938
84 	356   	0.416258	0.382651
85 	360   	0.416295	0.397471
86 	368   	0.416342	0.416342
87 	366   	0.415962	0.264366
88 	356   	0.415513	0.23657 
89 	362   	0.416297	0.398225
90 	355   	0.416341	0.415948
91 	360   	0.415991	0.324353
92 	358   	0.416103	0.320568
93 	370   	0.416196	0.357705
94 	352   	0.415649	0.263752
95 	380   	0.416221	0.379358
96 	360   	0.41621 	0.363274
97 	374   	0.416288	0.394587
98 	350   	0.416332	0.412348
99 	359   	0.415846	0.220265
100	363   	0.416207	0.362142
gen	nevals	avg     	min     
0  	0     	0.416207	0.362142
1  	362   	0.415753	0.223843
2  	370   	0.416076	0.309938
3  	367   	0.416342	0.416337
4  	374   	0.416342	0.416342
5  	362   	0.416342	0.416188
6  	360   	0.416246	0.377694
7  	358   	0.416342	0.416337
8  	368   	0.41634 	0.415263
9  	372   	0.415871	0.263752
10 	358   	0.41634 	0.415584
11 	363   	0.41584 	0.264366
12 	362   	0.416342	0.416342
13 	368   	0.4

In [9]:
for i in range(40):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.preprocessing import MinMaxScaler, StandardScaler
    from sklearn.utils import shuffle
    from sklearn import svm
    import pyswarms as ps
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import mean_absolute_error, r2_score
    from sklearn.svm import SVR
    from sklearn.model_selection import LeaveOneOut
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_error
    from sklearn.metrics import r2_score
    from deap import algorithms, base, creator, tools
    # Read data
    data = pd.read_excel('聚类分析2.xlsx')  # Please replace with your data file name
    data = data.drop('合金的牌号', axis=1)
    data = data.round(2)
    elements = ['Ni', 'Cr', 'Co', 'Fe', 'Al', 'Ti', 'Nb', 'Mo', 'W', 'C', 'B', 'Zr', '温度', '应力', '固溶处理温度',
                '固溶处理时间', '强化相溶解温度', '稳定时效温度', '稳定时效时间', '时效温度', '时效时间']

    X = np.zeros((data.shape[0], len(elements)))
    for i, row in data.iterrows():
        for j, el in enumerate(elements):
            X[i, j] = row[el]
    scaler = MinMaxScaler()
    X = scaler.fit_transform(X)
    Y = data[['蠕变时间']].values
    Y = np.log(Y)
    def evaluate(true_labels, pred_labels):
        errors = abs(pred_labels - true_labels)
        MAE = mean_absolute_error(true_labels, pred_labels)
        mape = 100 * np.mean(errors / true_labels)
        r2 = r2_score(true_labels, pred_labels)
        RMSE = mean_squared_error(true_labels, pred_labels, squared=False)
        return mape, MAE, RMSE, r2
    model_0 = SVR(C=37.722551111111111111, kernel='rbf', gamma=0.111111111111111111)
    cv = LeaveOneOut()
    Model = model_0
    X = X.reshape(-1, len(elements))
    Model.fit(X, Y)
    from sklearn.model_selection import cross_val_predict
    y_pred = cross_val_predict(Model, X, Y, cv=cv)
    model_accuracy1, model_MAE1, model_RMSE1, model_r21 = evaluate(Y, y_pred)
    print("R^2 score:", model_r21)
    def fitness_func(individual, X, Y, model):
        X_individual = np.tile(individual, (X.shape[0], 1))
        Y_pred = model.predict(X_individual)
        mse = mean_squared_error(Y, Y_pred)
        return (1 / (1 + mse),)
    # Define GA parameters
    POP_SIZE = 400
    NUM_GEN = 40
    CXPB = 0.9
    MUTPB = 0.01
    # Create a fitness function for maximizing fitness values
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)
    # Initialize toolbox
    toolbox = base.Toolbox()
    # Define attributes
    toolbox.register("attr_float", np.random.uniform, low=0, high=1)
    # Define individual and population
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=len(elements))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    # Define evaluation function
    toolbox.register("evaluate", fitness_func, X=X, Y=Y, model=Model)
    # Define genetic operators
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
    toolbox.register("select", tools.selTournament, tournsize=3)
    # Perform genetic algorithm
    population = toolbox.population(n=POP_SIZE)
    hall_of_fame = tools.HallOfFame(maxsize=1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    population, logbook = algorithms.eaSimple(population, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=NUM_GEN,
                                              stats=stats, halloffame=hall_of_fame, verbose=True)
    # Extract best individual and its fitness
    best_individual = hall_of_fame[0]
    best_fitness = best_individual.fitness.values[0]
    # Reshape best_individual
    best_particles = np.asarray(best_individual)
    print("------------------------------------------------------------------------------------")
    print("Best particles:", best_particles)
    print("Best fitness:", best_fitness)
    print("------------------------------------------------------------------------------------")

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

R^2 score: 0.8804390625989749
gen	nevals	avg     	min      
0  	400   	0.329482	0.0944736
1  	358   	0.367432	0.179861 
2  	360   	0.385129	0.141191 
3  	342   	0.387335	0.220437 
4  	355   	0.39246 	0.167813 
5  	364   	0.396089	0.235817 
6  	352   	0.395364	0.230162 
7  	366   	0.399393	0.219393 
8  	366   	0.399966	0.24496  
9  	360   	0.400225	0.194658 
10 	360   	0.401378	0.241058 
11 	361   	0.404155	0.252295 
12 	352   	0.403546	0.237818 
13 	364   	0.405989	0.264681 
14 	366   	0.407092	0.283446 
15 	368   	0.405798	0.26952  
16 	359   	0.404117	0.262168 
17 	356   	0.408299	0.316673 
18 	362   	0.405516	0.288012 
19 	369   	0.406452	0.275598 
20 	363   	0.407153	0.286702 
21 	339   	0.408922	0.312358 
22 	360   	0.409649	0.291374 
23 	374   	0.410552	0.349226 
24 	366   	0.411547	0.357386 
25 	363   	0.412844	0.300896 
26 	344   	0.412392	0.346984 
27 	368   	0.411983	0.328918 
28 	362   	0.413322	0.375762 
29 	354   	0.412193	0.255053 
30 	358   	0.414036	0.351002 
31 	382   

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

gen	nevals	avg     	min      
0  	400   	0.343733	0.0973219
1  	358   	0.368441	0.12631  
2  	362   	0.383056	0.154675 
3  	368   	0.391904	0.177356 
4  	362   	0.391307	0.187869 
5  	354   	0.393728	0.232557 
6  	375   	0.391176	0.157556 
7  	352   	0.399361	0.202723 
8  	366   	0.396048	0.238559 
9  	368   	0.393164	0.234511 
10 	366   	0.396808	0.253213 
11 	376   	0.398502	0.293389 
12 	368   	0.398338	0.218147 
13 	358   	0.401429	0.23526  
14 	363   	0.404463	0.250674 
15 	372   	0.405859	0.296488 
16 	349   	0.407424	0.313032 
17 	358   	0.406575	0.282784 
18 	376   	0.408363	0.320552 
19 	358   	0.405626	0.246533 
20 	360   	0.406945	0.283837 
21 	370   	0.407119	0.305932 
22 	366   	0.405787	0.217531 
23 	355   	0.404548	0.298462 
24 	363   	0.410138	0.341048 
25 	354   	0.407883	0.278011 
26 	360   	0.410736	0.296717 
27 	374   	0.409679	0.226974 
28 	363   	0.41124 	0.240503 
29 	366   	0.411926	0.241252 
30 	356   	0.41313 	0.346618 
31 	356   	0.412342	0.240927 
32 	352   

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

...

gen	nevals	avg     	min     
0  	400   	0.344961	0.125707
1  	367   	0.375964	0.205862
2  	363   	0.384294	0.1782  
3  	367   	0.388599	0.187946
4  	355   	0.394888	0.255499
5  	364   	0.389313	0.217681
6  	370   	0.394993	0.229693
7  	330   	0.394298	0.21172 
8  	376   	0.39124 	0.192351
9  	368   	0.3942  	0.21729 
10 	351   	0.396893	0.235543
11 	360   	0.396432	0.229055
12 	354   	0.394899	0.174445
13 	380   	0.395163	0.209736
14 	366   	0.401121	0.287668
15 	366   	0.401951	0.265889
16 	348   	0.401746	0.252209
17 	343   	0.401536	0.289559
18 	368   	0.40229 	0.311507
19 	366   	0.402261	0.284186
20 	359   	0.40324 	0.262085
21 	370   	0.402997	0.233549
22 	360   	0.405744	0.265517
23 	378   	0.404561	0.311406
24 	340   	0.405703	0.293206
25 	369   	0.404623	0.271802
26 	360   	0.407605	0.292491
27 	352   	0.4069  	0.299883
28 	358   	0.40581 	0.326106
29 	362   	0.407629	0.315746
30 	366   	0.408045	0.341988
31 	362   	0.408207	0.325447
32 	358   	0.410537	0.335133
33 	364   	0.4

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

gen	nevals	avg     	min     
0  	400   	0.339961	0.109601
1  	352   	0.375935	0.171951
2  	362   	0.388767	0.211584
3  	354   	0.387821	0.230119
4  	378   	0.394702	0.240967
5  	364   	0.389448	0.208609
6  	361   	0.393894	0.264619
7  	361   	0.396311	0.23666 
8  	364   	0.397722	0.210192
9  	359   	0.400406	0.247126
10 	358   	0.400614	0.222522
11 	340   	0.403115	0.263913
12 	363   	0.401336	0.265399
13 	364   	0.403877	0.292432
14 	360   	0.404786	0.330233
15 	358   	0.406421	0.290458
16 	362   	0.405339	0.312792
17 	366   	0.406531	0.318688
18 	356   	0.406942	0.30678 
19 	360   	0.408559	0.262876
20 	366   	0.407854	0.32894 
21 	368   	0.409006	0.309599
22 	376   	0.411082	0.350788
23 	352   	0.411952	0.329073
24 	362   	0.410085	0.320092
25 	372   	0.412469	0.350296
26 	355   	0.413129	0.375983
27 	364   	0.413515	0.22771 
28 	378   	0.414543	0.380189
29 	366   	0.414818	0.368693
30 	360   	0.415142	0.380989
31 	353   	0.415668	0.388718
32 	367   	0.415986	0.392768
33 	358   	0.4

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

gen	nevals	avg     	min     
0  	400   	0.337925	0.114656
1  	353   	0.375932	0.155222
2  	359   	0.382463	0.179835
3  	351   	0.385847	0.160213
4  	378   	0.386857	0.164396
5  	366   	0.384729	0.145308
6  	354   	0.391169	0.167566
7  	362   	0.392459	0.199905
8  	376   	0.391992	0.227301
9  	367   	0.395124	0.242386
10 	358   	0.396445	0.211274
11 	360   	0.394795	0.216215
12 	358   	0.39913 	0.223187
13 	355   	0.401758	0.234355
14 	367   	0.400559	0.184756
15 	356   	0.401111	0.189328
16 	364   	0.406876	0.275581
17 	370   	0.406063	0.189997
18 	346   	0.409254	0.359886
19 	354   	0.407857	0.297089
20 	358   	0.408495	0.32356 
21 	343   	0.406999	0.278202
22 	375   	0.40724 	0.290627
23 	358   	0.409826	0.34179 
24 	363   	0.410641	0.340289
25 	362   	0.411543	0.341078
26 	378   	0.410501	0.327439
27 	368   	0.409064	0.279879
28 	368   	0.411176	0.329501
29 	356   	0.412299	0.376157
30 	365   	0.412623	0.350456
31 	363   	0.41203 	0.339426
32 	356   	0.413169	0.362455
33 	355   	0.4